# Data Cleaning and Preprocessing Pipeline

This notebook consolidates what was done in EDA2 and feature selection so dataset generation is reproducible from one place.

Outputs created by this notebook:
- `Dataset/train_data_cleaned.csv` (EDA2-compatible cleaned dataset)
- `Dataset/train_data_feature_engineered_final.csv` (final dataset used by model training notebooks)

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

DATASET_DIR = Path("Dataset")
RAW_PATH = DATASET_DIR / "train_data.csv"
CLEANED_PATH = DATASET_DIR / "train_data_cleaned.csv"
FINAL_PATH = DATASET_DIR / "train_data_feature_engineered_final.csv"

df_raw = pd.read_csv(RAW_PATH)
print(f"Loaded raw data shape: {df_raw.shape}")

Loaded raw data shape: (11357, 78)


In [2]:
def count_supported_languages(value):
    if pd.isna(value):
        return 0
    cleaned = re.sub(r"<[^>]+>", " ", str(value))
    parts = [p.strip() for p in re.split(r"[,;|/]", cleaned) if p.strip()]
    return len(parts)


def safe_qcut_int(series, q=4, zero_as_zero=False):
    s = pd.to_numeric(series, errors="coerce")
    out = pd.Series(np.nan, index=s.index, dtype="float64")

    if zero_as_zero:
        out[:] = 0
        non_zero = s[s > 0].dropna()
        if non_zero.nunique() >= 2:
            bins = pd.qcut(
                non_zero,
                q=min(q, non_zero.nunique()),
                labels=False,
                duplicates="drop",
            )
            out.loc[bins.index] = bins.astype(float) + 1
        return out.fillna(0).astype(int)

    non_null = s.dropna()
    if non_null.nunique() < 2:
        return s.fillna(0).astype(int)

    bins = pd.qcut(
        non_null,
        q=min(q, non_null.nunique()),
        labels=False,
        duplicates="drop",
    )
    out.loc[bins.index] = bins.astype(float)
    return out.fillna(0).astype(int)

In [3]:
def build_cleaned_dataset_eda2(df_input):
    """Rebuild the cleaned dataset logic used in eda2.ipynb."""
    df = df_input.copy()

    text_and_id_cols = [
        'QueryID', 'QueryName', 'ResponseID', 'ResponseName', 'AboutText',
        'Background', 'ShortDescrip', 'DetailedDescrip', 'DRMNotice',
        'ExtUserAcctNotice', 'HeaderImage', 'LegalNotice', 'Reviews',
        'PCMinReqsText', 'PCRecReqsText', 'LinuxMinReqsText',
        'LinuxRecReqsText', 'MacMinReqsText', 'MacRecReqsText',
    ]
    reqs_cols = [
        'PCReqsHaveMin', 'PCReqsHaveRec', 'LinuxReqsHaveMin',
        'LinuxReqsHaveRec', 'MacReqsHaveMin', 'MacReqsHaveRec',
    ]

    present_text_id = [c for c in text_and_id_cols if c in df.columns]
    for col in present_text_id:
        df[col] = df[col].astype(str).str.strip()

    present_reqs = [c for c in reqs_cols if c in df.columns]
    for col in present_reqs:
        df[col] = df[col].fillna(False).astype(bool).astype(int)

    if 'ReleaseDate' in df.columns:
        df['ReleaseDate'] = pd.to_datetime(df['ReleaseDate'], errors='coerce')

    drop_stage_1 = present_text_id + (['ReleaseDate'] if 'ReleaseDate' in df.columns else []) + present_reqs
    df = df.drop(columns=drop_stage_1, errors='ignore')

    if 'RequiredAge' in df.columns:
        df = df.drop(columns=['RequiredAge'], errors='ignore')

    if 'DemoCount' in df.columns:
        demo = pd.to_numeric(df['DemoCount'], errors='coerce').fillna(0)
        df['HasDemo'] = (demo > 0).astype(int)

    if 'DeveloperCount' in df.columns:
        dev = pd.to_numeric(df['DeveloperCount'], errors='coerce').fillna(0)
        df['MultipleDevs'] = (dev > 1).astype(int)

    if 'DLCCount' in df.columns:
        dlc = pd.to_numeric(df['DLCCount'], errors='coerce').fillna(0)
        df['DLCCountCat'] = np.where(dlc > 1, 2, dlc).astype(int)

    if 'Metacritic' in df.columns:
        df['Metacritic_Bins'] = safe_qcut_int(df['Metacritic'], q=4, zero_as_zero=True)

    if 'MovieCount' in df.columns:
        df['MovieCountCat'] = safe_qcut_int(df['MovieCount'], q=4, zero_as_zero=False)

    if 'PackageCount' in df.columns:
        package_count = pd.to_numeric(df['PackageCount'], errors='coerce').fillna(0)
        df['PackageCategory'] = np.where(package_count > 1, '> 1', package_count.astype(int).astype(str))

    if 'PublisherCount' in df.columns:
        publisher_count = pd.to_numeric(df['PublisherCount'], errors='coerce').fillna(0)
        df['PublisherCategory'] = np.where(publisher_count > 1, '> 1', '<= 1')

    if {'PriceFinal', 'PriceInitial'}.issubset(df.columns):
        price_final = pd.to_numeric(df['PriceFinal'], errors='coerce').fillna(0)
        price_initial = pd.to_numeric(df['PriceInitial'], errors='coerce').fillna(0)
        df['HasDiscount'] = (price_final < price_initial).astype(int)

    if 'SupportedLanguages' in df.columns:
        lang_series = df['SupportedLanguages'].fillna('')
        df['HasEnglish'] = lang_series.str.contains('english', case=False, na=False).astype(int)
        df['LanguageCount'] = lang_series.apply(count_supported_languages).astype(int)

    if 'SupportURL' in df.columns:
        df['HasSupportURL'] = (df['SupportURL'].fillna('').str.strip() != '').astype(int)

    if 'SupportEmail' in df.columns:
        df['HasSupportEmail'] = (df['SupportEmail'].fillna('').str.strip() != '').astype(int)

    if 'Website' in df.columns:
        websites = df['Website'].fillna('').str.strip().str.lower().str.rstrip('/')
        df['IsComWebsite'] = websites.str.endswith('.com').astype(int)
        df['IsHttpWebsite'] = websites.str.startswith('http').astype(int)

    df = df.drop(
        columns=['AchievementHighlightedCount', 'FreeVerAvail', 'SubscriptionAvail'],
        errors='ignore',
    )

    bool_cols = [c for c in ['PurchaseAvail', 'IsFree', 'ControllerSupport'] if c in df.columns]
    for col in bool_cols:
        df[col] = df[col].fillna(False).astype(bool).astype(int)

    text_and_link_cols = ['Website', 'SupportURL', 'SupportEmail', 'PriceCurrency', 'SupportedLanguages']
    drop_stage_2 = [c for c in text_and_link_cols + bool_cols + ['ScreenshotCount'] if c in df.columns]
    df = df.drop(columns=drop_stage_2, errors='ignore')

    remaining_bool_cols = df.select_dtypes(include='bool').columns.tolist()
    if remaining_bool_cols:
        df[remaining_bool_cols] = df[remaining_bool_cols].astype(int)

    return df

In [4]:
def build_final_training_dataset(df_input, drop_problematic=False):
    """Build the final engineered dataset guided by feature_selection experiments."""
    target = 'RecommendationCount'
    df = df_input.copy()

    if 'ReleaseDate' in df.columns:
        dt = pd.to_datetime(df['ReleaseDate'], errors='coerce')
        df['ReleaseYear'] = dt.dt.year
        df['ReleaseMonth'] = dt.dt.month
        df['ReleaseDayOfWeek'] = dt.dt.dayofweek
        df['DaysSinceRelease'] = (pd.Timestamp('2017-01-01') - dt).dt.days

    if 'DemoCount' in df.columns:
        demo = pd.to_numeric(df['DemoCount'], errors='coerce').fillna(0)
        df['HasDemo'] = (demo > 0).astype(int)

    if 'DeveloperCount' in df.columns:
        dev = pd.to_numeric(df['DeveloperCount'], errors='coerce').fillna(0)
        df['MultipleDevs'] = (dev > 1).astype(int)

    if 'DLCCount' in df.columns:
        dlc = pd.to_numeric(df['DLCCount'], errors='coerce').fillna(0)
        df['DLCCountCat'] = np.where(dlc > 1, 2, dlc).astype(int)

    if 'Metacritic' in df.columns:
        df['Metacritic_Bins'] = safe_qcut_int(df['Metacritic'], q=4, zero_as_zero=True)

    if 'MovieCount' in df.columns:
        df['MovieCountCat'] = safe_qcut_int(df['MovieCount'], q=4, zero_as_zero=False)

    if 'PackageCount' in df.columns:
        package_count = pd.to_numeric(df['PackageCount'], errors='coerce').fillna(0)
        df['PackageCountCat'] = np.where(package_count > 1, 2, package_count).astype(int)

    if 'PublisherCount' in df.columns:
        publisher_count = pd.to_numeric(df['PublisherCount'], errors='coerce').fillna(0)
        df['PublisherCountCat'] = (publisher_count > 1).astype(int)

    if {'PriceInitial', 'PriceFinal'}.issubset(df.columns):
        price_initial = pd.to_numeric(df['PriceInitial'], errors='coerce').fillna(0)
        price_final = pd.to_numeric(df['PriceFinal'], errors='coerce').fillna(0)

        df['HasDiscount'] = (price_final < price_initial).astype(int)
        df['DiscountAmt'] = price_initial - price_final
        df['DiscountPct'] = np.where(price_initial > 0, (price_initial - price_final) / price_initial, 0.0)
        df['IsFreeFinal'] = (price_final == 0).astype(int)

    if 'SupportedLanguages' in df.columns:
        lang_series = df['SupportedLanguages'].fillna('')
        df['HasEnglish'] = lang_series.str.contains('english', case=False, na=False).astype(int)
        df['LanguageCount'] = lang_series.apply(count_supported_languages).astype(int)

    for col in ['AboutText', 'Background', 'ShortDescrip', 'DetailedDescrip', 'Reviews', 'LegalNotice']:
        if col in df.columns:
            df[f'{col}_Len'] = df[col].fillna('').astype(str).str.len()

    if 'Website' in df.columns:
        website_clean = df['Website'].fillna('').astype(str).str.strip().str.lower()
        df['HasWebsite'] = website_clean.ne('').astype(int)
        df['IsComWebsite'] = website_clean.str.rstrip('/').str.endswith('.com').astype(int)
        df['IsHttpWebsite'] = website_clean.str.startswith('http').astype(int)

    if 'SupportEmail' in df.columns:
        support_email = df['SupportEmail'].fillna('').astype(str).str.strip()
        df['HasSupportEmail'] = support_email.ne('').astype(int)

    if 'SupportURL' in df.columns:
        support_url = df['SupportURL'].fillna('').astype(str).str.strip()
        df['HasSupportURL'] = support_url.ne('').astype(int)

    if 'DRMNotice' in df.columns:
        df['HasDRMNotice'] = df['DRMNotice'].fillna('').astype(str).str.strip().ne('').astype(int)

    if 'ExtUserAcctNotice' in df.columns:
        df['HasExtUserAcct'] = df['ExtUserAcctNotice'].fillna('').astype(str).str.strip().ne('').astype(int)

    if 'LegalNotice' in df.columns:
        df['HasLegalNotice'] = df['LegalNotice'].fillna('').astype(str).str.strip().ne('').astype(int)

    genre_cols = [c for c in df.columns if c.startswith('GenreIs')]
    if genre_cols:
        df['GenreCount'] = df[genre_cols].fillna(False).astype(bool).astype(int).sum(axis=1)

    category_cols = [c for c in df.columns if c.startswith('Category')]
    if category_cols:
        df['CategoryCount'] = df[category_cols].fillna(False).astype(bool).astype(int).sum(axis=1)

    platform_cols = [c for c in ['PlatformWindows', 'PlatformLinux', 'PlatformMac'] if c in df.columns]
    if platform_cols:
        df['PlatformCount'] = df[platform_cols].fillna(False).astype(bool).astype(int).sum(axis=1)

    reqs_cols = [
        'PCReqsHaveMin', 'PCReqsHaveRec',
        'LinuxReqsHaveMin', 'LinuxReqsHaveRec',
        'MacReqsHaveMin', 'MacReqsHaveRec',
    ]
    present_reqs = [c for c in reqs_cols if c in df.columns]
    if present_reqs:
        df['ReqsCount'] = df[present_reqs].fillna(False).astype(bool).astype(int).sum(axis=1)

    heavy_tail_cols = [
        'SteamSpyOwners', 'SteamSpyOwnersVariance',
        'SteamSpyPlayersEstimate', 'SteamSpyPlayersVariance',
        'AchievementCount', 'DLCCount', 'MovieCount', 'ScreenshotCount',
        'PriceInitial', 'PriceFinal',
    ]
    for col in heavy_tail_cols:
        if col in df.columns:
            values = pd.to_numeric(df[col], errors='coerce').fillna(0).clip(lower=0)
            df[f'{col}_log'] = np.log1p(values)

    if {'SteamSpyPlayersEstimate', 'SteamSpyOwners'}.issubset(df.columns):
        owners = pd.to_numeric(df['SteamSpyOwners'], errors='coerce').replace(0, np.nan)
        players = pd.to_numeric(df['SteamSpyPlayersEstimate'], errors='coerce')
        ratio = players / owners
        df['PlayersToOwnersRatio'] = ratio.replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 10)

    if {'AchievementCount', 'PriceFinal'}.issubset(df.columns):
        achievement = pd.to_numeric(df['AchievementCount'], errors='coerce')
        price_final = pd.to_numeric(df['PriceFinal'], errors='coerce')
        df['AchievementsPerPrice'] = achievement / (price_final + 1)

    if {'MovieCount', 'ScreenshotCount'}.issubset(df.columns):
        movie = pd.to_numeric(df['MovieCount'], errors='coerce')
        screenshot = pd.to_numeric(df['ScreenshotCount'], errors='coerce')
        df['MoviesPerScreenshot'] = movie / (screenshot + 1)

    drop_cols = [
        'QueryID', 'ResponseID', 'QueryName', 'ResponseName',
        'AboutText', 'Background', 'ShortDescrip', 'DetailedDescrip',
        'HeaderImage', 'LegalNotice', 'Reviews', 'SupportedLanguages',
        'SupportEmail', 'SupportURL', 'Website',
        'PCMinReqsText', 'PCRecReqsText',
        'LinuxMinReqsText', 'LinuxRecReqsText',
        'MacMinReqsText', 'MacRecReqsText',
        'ReleaseDate',
    ]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

    problematic_features = [
        'SteamSpyOwners', 'SteamSpyOwnersVariance',
        'SteamSpyPlayersEstimate', 'SteamSpyPlayersVariance',
        'SteamSpyOwners_log', 'SteamSpyOwnersVariance_log',
        'SteamSpyPlayersEstimate_log', 'SteamSpyPlayersVariance_log',
        'DRMNotice', 'ExtUserAcctNotice',
    ]
    if drop_problematic:
        df = df.drop(columns=[c for c in problematic_features if c in df.columns], errors='ignore')

    bool_cols = df.select_dtypes(include='bool').columns.tolist()
    if bool_cols:
        df[bool_cols] = df[bool_cols].astype(int)

    cat_cols = [c for c in ['PriceCurrency', 'DRMNotice', 'ExtUserAcctNotice'] if c in df.columns]
    for col in cat_cols:
        df[col] = df[col].fillna('missing').astype(str).str.strip()
        df.loc[df[col] == '', col] = 'missing'

    if target in df.columns:
        df[target] = pd.to_numeric(df[target], errors='coerce').fillna(0)

    numeric_cols = [c for c in df.columns if c not in cat_cols + [target]]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    if numeric_cols:
        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
        medians = df[numeric_cols].median(numeric_only=True)
        df[numeric_cols] = df[numeric_cols].fillna(medians).fillna(0)

    return df

In [5]:
df_cleaned = build_cleaned_dataset_eda2(df_raw)
df_cleaned.to_csv(CLEANED_PATH, index=False)

DROP_PROBLEMATIC_FEATURES = False
df_final = build_final_training_dataset(
    df_raw,
    drop_problematic=DROP_PROBLEMATIC_FEATURES,
)
df_final.to_csv(FINAL_PATH, index=False)

print(f"Saved cleaned dataset : {CLEANED_PATH} | shape={df_cleaned.shape}")
print(f"Saved final dataset   : {FINAL_PATH} | shape={df_final.shape}")
print(f"drop_problematic set to: {DROP_PROBLEMATIC_FEATURES}")

Saved cleaned dataset : Dataset/train_data_cleaned.csv | shape=(11357, 53)
Saved final dataset   : Dataset/train_data_feature_engineered_final.csv | shape=(11357, 104)
drop_problematic set to: False


In [6]:
target = 'RecommendationCount'
missing_cleaned = int(df_cleaned.isna().sum().sum())
missing_final = int(df_final.isna().sum().sum())
object_cols_final = [c for c in df_final.columns if df_final[c].dtype == 'object']

print('Quality checks')
print('-' * 60)
print(f'Missing values (cleaned): {missing_cleaned}')
print(f'Missing values (final)  : {missing_final}')
print(f'Target exists in final  : {target in df_final.columns}')
print(f'Object columns in final : {object_cols_final}')

display(df_final.head(3))

Quality checks
------------------------------------------------------------
Missing values (cleaned): 0
Missing values (final)  : 0
Target exists in final  : True
Object columns in final : []


,RequiredAge,DemoCount,DeveloperCount,DLCCount,Metacritic,MovieCount,PackageCount,RecommendationCount,PublisherCount,ScreenshotCount,SteamSpyOwners,SteamSpyOwnersVariance,SteamSpyPlayersEstimate,SteamSpyPlayersVariance,AchievementCount,AchievementHighlightedCount,ControllerSupport,IsFree,FreeVerAvail,PurchaseAvail,SubscriptionAvail,PlatformWindows,PlatformLinux,PlatformMac,PCReqsHaveMin,PCReqsHaveRec,LinuxReqsHaveMin,LinuxReqsHaveRec,MacReqsHaveMin,MacReqsHaveRec,CategorySinglePlayer,CategoryMultiplayer,CategoryCoop,CategoryMMO,CategoryInAppPurchase,CategoryIncludeSrcSDK,CategoryIncludeLevelEditor,CategoryVRSupport,GenreIsNonGame,GenreIsIndie,GenreIsAction,GenreIsAdventure,GenreIsCasual,GenreIsStrategy,GenreIsRPG,GenreIsSimulation,GenreIsEarlyAccess,GenreIsFreeToPlay,GenreIsSports,GenreIsRacing,GenreIsMassivelyMultiplayer,PriceCurrency,PriceInitial,PriceFinal,DRMNotice,ExtUserAcctNotice,ReleaseYear,ReleaseMonth,ReleaseDayOfWeek,DaysSinceRelease,HasDemo,MultipleDevs,DLCCountCat,Metacritic_Bins,MovieCountCat,PackageCountCat,PublisherCountCat,HasDiscount,DiscountAmt,DiscountPct,IsFreeFinal,HasEnglish,LanguageCount,AboutText_Len,Background_Len,ShortDescrip_Len,DetailedDescrip_Len,Reviews_Len,LegalNotice_Len,HasWebsite,IsComWebsite,IsHttpWebsite,HasSupportEmail,HasSupportURL,HasDRMNotice,HasExtUserAcct,HasLegalNotice,GenreCount,CategoryCount,PlatformCount,ReqsCount,SteamSpyOwners_log,SteamSpyOwnersVariance_log,SteamSpyPlayersEstimate_log,SteamSpyPlayersVariance_log,AchievementCount_log,DLCCount_log,MovieCount_log,ScreenshotCount_log,PriceInitial_log,PriceFinal_log,PlayersToOwnersRatio,AchievementsPerPrice,MoviesPerScreenshot
0,0,0,1,0,88,0,1,68991,1,13,13033334,92789,9140731,78136,0,0,0,0,0,1,0,1,1,1,1,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,USD,9.99,9.99,missing,missing,2000.0,11.0,2.0,5905.0,0,0,0,4,0,1,0,0,0.0,0.0,0,1,1,312,86,1,312,1,1,0,0,0,0,1,0,0,0,1,1,3,3,16.383021,11.438094,16.028251,11.266219,0.0,0.0,0.0,2.639057,2.396986,2.396986,0.701335,0.0,0.0
1,0,0,1,0,0,0,1,2439,1,5,5399140,60368,753627,22699,0,0,0,0,0,1,0,1,1,1,1,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,USD,4.99,4.99,missing,missing,1999.0,4.0,3.0,6485.0,0,0,0,0,0,1,0,0,0.0,0.0,0,1,1,330,86,1,330,1,1,0,0,0,0,0,0,0,0,1,1,3,3,15.501750,11.008231,13.532654,10.030120,0.0,0.0,0.0,1.791759,1.790091,1.790091,0.139583,0.0,0.0
2,0,0,1,0,79,0,1,2319,1,5,7621102,71499,1709740,34145,0,0,0,0,0,1,0,1,1,1,1,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,USD,4.99,4.99,missing,missing,2003.0,5.0,3.0,4994.0,0,0,0,3,0,1,0,0,0.0,0.0,0,1,1,424,86,1,424,1,1,1,1,1,0,0,0,0,0,1,1,3,3,15.846432,11.177453,14.351852,10.438401,0.0,0.0,0.0,1.791759,1.790091,1.790091,0.224343,0.0,0.0


## Notes

- Keep `DROP_PROBLEMATIC_FEATURES = False` if you want compatibility with the current saved final dataset pattern from feature selection.
- Set `DROP_PROBLEMATIC_FEATURES = True` to reproduce the alternative experiment that removes potentially unstable SteamSpy and notice-based features.
- The final training notebook should keep reading `Dataset/train_data_feature_engineered_final.csv`.